In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# **1. Imports & Setup**

In [ ]:
import os
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F

import torchvision.transforms as transforms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

# **2. Dataset Class**

In [ ]:
class LeafDataset(Dataset):
    def __init__(self, lr_dir, hr_dir=None):
        self.lr_dir = lr_dir
        self.hr_dir = hr_dir
        self.files = sorted(os.listdir(lr_dir))
        self.transform = transforms.Compose([transforms.ToTensor()])

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        file = self.files[idx]
        lr = Image.open(os.path.join(self.lr_dir, file)).convert("RGB")
        lr = self.transform(lr)

        if self.hr_dir:
            hr = Image.open(os.path.join(self.hr_dir, file)).convert("RGB")
            hr = self.transform(hr)
            return lr, hr
        else:
            return lr, file

# **3. Model (U-Net Style Generator)**

In [ ]:
class ResBlock(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.block = nn.Sequential(nn.Conv2d(c, c, 3, padding=1), 
                                   nn.BatchNorm2d(c),
                                   nn.PReLU(),
                                   nn.Conv2d(c, c, 3, padding=1),
                                   nn.BatchNorm2d(c)
                                  )
    def forward(self, x):
        return x + self.block(x)

class ConvBlock(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1),
            nn.BatchNorm2d(out_c),
            nn.PReLU(),
            nn.Conv2d(out_c, out_c, 3, padding=1),
            nn.BatchNorm2d(out_c),
            nn.PReLU()
        )
    def forward(self, x):
        return self.block(x)

class PixelShuffleUp(nn.Module):
    def __init__(self, in_c, out_c, scale=2):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_c, out_c*(scale**2), 3, padding=1),
            nn.PixelShuffle(scale),
            nn.PReLU()
        )
    def forward(self, x):
        return self.block(x)


class UNetSR(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc1 = ConvBlock(3, 64)
        self.enc2 = ConvBlock(64, 128)
        self.pool = nn.MaxPool2d(2)
        self.bottleneck = ConvBlock(128, 512)
        self.res1 = ResBlock(512)
        self.res2 = ResBlock(512)
        self.res3 = ResBlock(512)
        self.up1  = PixelShuffleUp(512, 128, scale=2)
        self.dec1 = ConvBlock(256, 128)   
        self.up2  = PixelShuffleUp(128, 64, scale=2)
        self.dec2 = ConvBlock(128, 64)    
        self.up3  = PixelShuffleUp(64, 64, scale=2)
        self.up4  = PixelShuffleUp(64, 64, scale=2)
        self.final = nn.Conv2d(64, 3, 1)

    def forward(self,x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        b = self.bottleneck(self.pool(e2))
        b = self.res1(b)
        b = self.res2(b)
        b = self.res3(b)
        
        d1 = self.up1(b)
        d1 = self.dec1(torch.cat([d1,e2], dim=1))
        d2 = self.up2(d1)
        d2 = self.dec2(torch.cat([d2, e1], dim=1))
        d3 = self.up3(d2)
        d4 = self.up4(d3)
        out = torch.sigmoid(self.final(d4))

        return out

In [ ]:
class PatchDiscriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 64, 4, 2, 1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(64, 128, 4, 2,1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(128, 256, 4, 2,1),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(256, 1, 4, 1,1),
        )
    def forward(self, x):
        return self.net(x)

In [ ]:
from torchvision.models import vgg19
vgg_full = vgg19(weights=None)
vgg_full.load_state_dict(
    torch.load("/kaggle/input/competitions/plant-leaves-super-resolution-challenge/vgg19_weights.pth")
)

vgg_feat = vgg_full.features[:16].to(device).eval()

for p in vgg_feat.parameters():
    p.requires_grad=False

def perceptual_loss(pred, target):
    return F.mse_loss(vgg_feat(pred), vgg_feat(target))

# **4. Loading Data**

In [ ]:
train_lr = "/kaggle/input/competitions/plant-leaves-super-resolution-challenge/train_Low_Resolution"
train_hr = "/kaggle/input/competitions/plant-leaves-super-resolution-challenge/train_High_Resolution"
test_lr = "/kaggle/input/competitions/plant-leaves-super-resolution-challenge/test_Low_Resolution"

train_dataset = LeafDataset(train_lr, train_hr)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)

test_dataset = LeafDataset(test_lr)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

# **5. Model Initialization**

In [ ]:
generator = UNetSR().to(device)
discriminator = PatchDiscriminator().to(device)

criterion = nn.L1Loss()

opt_g = optim.Adam(generator.parameters(), lr=1e-4, betas=(0.9, 0.999))
opt_d = optim.Adam(generator.parameters(), lr=1e-4, betas=(0.9, 0.999))

scheduler_g = torch.optim.lr_scheduler.CosineAnnealingLR(opt_g, T_max=80)
scheduler_d = torch.optim.lr_scheduler.CosineAnnealingLR(opt_g, T_max=80)

# **6. Training Loop**

In [ ]:
epochs = 80

for epoch in range(epochs):
    generator.train()
    discriminator.train()
    total_g, total_d = 0, 0

    for lr_img, hr_img in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}"):
        lr_img, hr_img = lr_img.to(device), hr_img.to(device)

        fake = generator(lr_img).detach()
        d_real = discriminator(hr_img)
        d_fake = discriminator(fake)
        d_loss = (F.relu(1.0-d_real) + F.relu(1.0+d_fake )).mean()

        
        opt_d.zero_grad()
        d_loss.backward()
        opt_d.step()

        pred=generator(lr_img)
        g_pixel = criterion(pred, hr_img)
        g_percep = perceptual_loss(pred, hr_img)
        g_adv = -discriminator(pred).mean()
        g_loss = 1.0*g_pixel + 0.006*g_percep + 0.001*g_adv

        opt_g.zero_grad()
        g_loss.backward()
        torch.nn.utils.clip_grad_norm_(generator.parameters(), 1.0)
        opt_g.step()

        total_g += g_loss.item()
        total_d += d_loss.item()

    scheduler_g.step()
    scheduler_d.step()

    print(f"Epoch {epoch+1}, G Loss: {total_g/len(train_loader):.4f} | D Loss: {total_d/len(train_loader):.4f}")

# **7. Inference**

In [ ]:
def tta_predict(model, lr):
    preds=[]
    for k in range(4):
        rot = torch.rot90(lr, k, dims=[2,3])
        p = model(rot)
        preds.append(torch.rot90(p, -k, dims=[2,3]))

        rot_f = torch.flip(rot, dims=[3])
        p_f = model(rot_f)
        preds.append(torch.rot90(torch.flip(p_f, dims=[3]), -k, dims=[2,3]))

    return torch.stack(preds).mean(0)

generator.eval()
results = []

with torch.no_grad():
    for lr_img, fname in tqdm(test_loader):
        lr_img = lr_img.to(device)
        pred = tta_predict(generator, lr_img)

        pred = pred[0].cpu().numpy()
        pred = np.transpose(pred, (1,2,0))
        pred = (pred * 255).clip(0,255).astype(np.uint8)
        flat = pred.flatten()
        pixel_str = " ".join(map(str, flat))
        results.append([fname[0], pixel_str])

# **Final Submission**

In [ ]:
submission_df = pd.DataFrame(results, columns=['Id', 'Pixels'])
submission_df.to_csv("submission.csv", index=False)
print(submission_df.head())
print(f"Total rows: {len(submission_df)}")